In [ ]:
"""
07c - Upload the result rasters to Earth Engine for visualisation

Uploads the five rasters written by notebook 07b as Earth Engine Image assets, so they
can be inspected against a Sentinel-2 base map in the Code Editor.

Prerequisite: notebook 07b has completed and these files exist under
outputs/inference_full_roi/:
    severity_30m.tif  persistence_30m.tif  recovery_gap_30m.tif
    recurrence_30m.tif  burned_prob_30m.tif

This step is for visual inspection only. It produces no thesis result.
Runtime: 10 to 25 minutes, depending on file size and connection.
"""

'\nNotebook 07c v4 — Upload result rasters to GEE for visualisation\n\nPurpose:\n    Guide the upload of the 5 result rasters from Notebook 07b to GEE\n    as Image assets so they can be visualised together with a Sentinel-2\n    base map in the GEE Code Editor.\n\nPrerequisites:\n    Notebook 07b v4 has completed. The following files exist in\n    MyDrive/thesisv4/outputs/inference_full_roi/:\n        - severity_30m.tif\n        - persistence_30m.tif\n        - recovery_gap_30m.tif\n        - recurrence_30m.tif\n        - burned_prob_30m.tif       (new in v3)\n\nRuntime: 10-25 minutes (upload depends on file size and connection).\n'

# 07c - Upload results to Earth Engine

**Purpose** - publish the 30 m result rasters as Earth Engine assets for visual
inspection against a Sentinel-2 base map.

**Main inputs** - the rasters written by notebook 07b.

**Main outputs** - Earth Engine Image assets and `gee_asset_info.json`.

**Role in the thesis workflow** - visualisation only. No thesis number, table or
figure depends on this notebook; it is included so the 30 m chain is complete.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import ee
import json
import time
from pathlib import Path
from datetime import datetime

try:
    ee.Initialize(project="bar-project-457815")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="bar-project-457815")

print("GEE initialised.")

GEE initialised.


## 2. Locate the result rasters

In [ ]:
RESULTS_DIR = Path("/content/drive/MyDrive/thesisv4/outputs/inference_full_roi")

result_files = {
    "severity":     RESULTS_DIR / "severity_30m.tif",
    "persistence":  RESULTS_DIR / "persistence_30m.tif",
    "recovery_gap": RESULTS_DIR / "recovery_gap_30m.tif",
    "recurrence":   RESULTS_DIR / "recurrence_30m.tif",
    "burned_prob":  RESULTS_DIR / "burned_prob_30m.tif",
}

for name, path in result_files.items():
    assert path.exists(), f"Missing: {path}"
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"  {name:15s} {path.name} ({size_mb:.1f} MB)")

  severity        severity_30m.tif (16.1 MB)
  persistence     persistence_30m.tif (16.1 MB)
  recovery_gap    recovery_gap_30m.tif (16.1 MB)
  recurrence      recurrence_30m.tif (16.1 MB)
  burned_prob     burned_prob_30m.tif (16.1 MB)


## 3. Prepare asset paths

In [ ]:
TIMESTAMP    = datetime.now().strftime("%Y%m%d_%H%M")
ASSET_PREFIX = f"projects/bar-project-457815/assets/ris_inference_v4_{TIMESTAMP}"

print(f"Assets will be named: {ASSET_PREFIX}_<dimension>")

Assets will be named: projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_<dimension>


## 4. Manual upload - the recommended route

The Python API has no direct upload-from-Drive function, so the practical route is
manual, through the Earth Engine Code Editor:

1. Open the Code Editor at https://code.earthengine.google.com/
2. Open the **Assets** tab on the left
3. Choose **NEW - Image upload (GeoTIFF)**
4. Select the file from Drive (download it first if the browser upload dialog does
   not offer a "from Drive" option)
5. Set the asset name to the value listed below

In [ ]:
print("Manual upload paths (files are on Drive):")
for name, path in result_files.items():
    print(f"  {name:15s}: {path}")

print("\nAsset names to use in the GEE upload dialog:")
for name in result_files:
    print(f"  {ASSET_PREFIX}_{name}")

## 5. Alternative - automated upload through Cloud Storage

If a Cloud Storage bucket is available to the project, this cell uploads each raster
to the bucket and ingests it as an Earth Engine asset. Skip it if you are using the
manual route above.

In [ ]:
# Uncomment and configure if using GCS:
#
# GCS_BUCKET = "your-bucket-name"
#
# from google.cloud import storage
# client = storage.Client(project="bar-project-457815")
# bucket = client.bucket(GCS_BUCKET)
#
# for name, local_path in result_files.items():
#     gcs_blob = f"thesis_uploads/{name}_30m_{TIMESTAMP}.tif"
#     blob = bucket.blob(gcs_blob)
#     print(f"Uploading {name} to gs://{GCS_BUCKET}/{gcs_blob}...")
#     blob.upload_from_filename(local_path)
#
#     asset_id   = f"{ASSET_PREFIX}_{name}"
#     request_id = ee.data.newTaskId()[0]
#     params = {
#         "id":      asset_id,
#         "sources": [{"uris": [f"gs://{GCS_BUCKET}/{gcs_blob}"]}],
#     }
#     ee.data.startIngestion(request_id, params)
#     print(f"  Ingestion started for {asset_id}")

## 6. Save the asset metadata for the visualisation script

In [ ]:
asset_info = {
    "timestamp":    TIMESTAMP,
    "asset_prefix": ASSET_PREFIX,
    "expected_assets": [f"{ASSET_PREFIX}_{name}"
                        for name in result_files.keys()],
}

with open(RESULTS_DIR / "gee_asset_info.json", "w") as f:
    json.dump(asset_info, f, indent=2)

print(json.dumps(asset_info, indent=2))
print(f"\nSaved to: {RESULTS_DIR / 'gee_asset_info.json'}")
print("\nAfter uploading, paste the JS visualisation script into the GEE")
print("Code Editor and update TIMESTAMP to match the value above.")

{
  "timestamp": "20260813_0934",
  "asset_prefix": "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934",
  "expected_assets": [
    "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_severity",
    "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_persistence",
    "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_recovery_gap",
    "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_recurrence",
    "projects/bar-project-457815/assets/ris_inference_v4_20260813_0934_burned_prob"
  ]
}

Saved to: /content/drive/MyDrive/thesisv4/outputs/inference_full_roi/gee_asset_info.json

After uploading, paste the JS visualisation script into the GEE
Code Editor and update TIMESTAMP to match the value above.


In [1]:
"""
sensitivity_analyses.py

Runs two sensitivity analyses:

1. Recovery-gap window sensitivity:
   - 6-month windows (Jan-Jun 2017 vs Jul-Dec 2025)
   - 24-month windows (2017-2018 vs 2024-2025)
   - Compare to 12-month main setup

2. Composite RIS weighting sensitivity:
   - 5 profiles: equal, severity-dominant, recurrence-dominant,
     persistence-dominant, recovery-gap-dominant
   - Compute pairwise Jaccard overlap of top-10% pixels
   - Compute Spearman rank correlation between profiles

Prerequisites:
    ~/thesisv4/data/processed/X_scaled.npy
    ~/thesisv4/data/processed/y_burned.npy
    ~/thesisv4/data/processed/feature_names.json
    ~/thesisv4/data/processed/splits.json
    ~/thesisv4/models/main/predictions_test.npz  # test-set model predictions

Outputs:
    ~/thesisv4/outputs/tables/recovery_gap_sensitivity.csv
    ~/thesisv4/outputs/tables/composite_weighting_sensitivity.csv
"""

import os, sys, json
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr

THESIS_ROOT = os.path.expanduser("~/thesisv4")
sys.path.insert(0, THESIS_ROOT)

# ============================================================
# 1. Recovery-gap window sensitivity
# ============================================================
print("=" * 60)
print("1. Recovery-gap window sensitivity")
print("=" * 60)

# Load raw NBR (needed for recovery_gap computation)
X = np.load(os.path.join(THESIS_ROOT, "data/processed/X_scaled.npy"))
y = np.load(os.path.join(THESIS_ROOT, "data/processed/y_burned.npy"))

with open(os.path.join(THESIS_ROOT, "data/processed/feature_names.json")) as f:
    feature_names = json.load(f)

nbr_idx = feature_names.index("NBR")
print(f"NBR feature index: {nbr_idx}")
print(f"X shape: {X.shape}")

# Note: X_scaled uses standardized features. We need the raw NBR values
# to compute recovery_gap. If nbr_raw is not available, we'll compute
# from X_scaled by inverting the scaler. Adjust here if you have nbr_raw
# stored separately.
nbr_raw_path = os.path.join(THESIS_ROOT, "data/processed/nbr_raw.npy")
if os.path.exists(nbr_raw_path):
    nbr_raw = np.load(nbr_raw_path)
    print(f"Loaded nbr_raw from disk: {nbr_raw.shape}")
else:
    print("Warning: nbr_raw.npy not found; using X_scaled[:,:,NBR] (standardized)")
    print("Consider re-loading from the raw CSV to get true NBR values")
    nbr_raw = X[:, :, nbr_idx]  # This is standardized, not raw!


def compute_recovery_gap(nbr_raw, y, baseline_months, recent_months):
    """Compute recovery_gap with configurable window sizes."""
    N, T = nbr_raw.shape
    recovery_gap = np.zeros(N, dtype=np.float32)
    for i in range(N):
        burned_months = np.where(y[i] == 1)[0]
        if len(burned_months) == 0:
            continue
        baseline_nbr = float(np.nanmean(nbr_raw[i, :baseline_months]))
        recent_nbr   = float(np.nanmean(nbr_raw[i, -recent_months:]))
        gap = (baseline_nbr - recent_nbr) / (abs(baseline_nbr) + 1e-6)
        recovery_gap[i] = float(np.clip(gap, 0, 1))
    return recovery_gap


# Compute recovery_gap under three window configurations
configs = [
    ("6-month",  6,  6),
    ("12-month (main)", 12, 12),
    ("24-month", 24, 24),
]

results = {}
for label, base_m, rec_m in configs:
    rg = compute_recovery_gap(nbr_raw, y, base_m, rec_m)
    results[label] = rg
    print(f"\n{label} windows (baseline={base_m}, recent={rec_m}):")
    print(f"  Mean: {rg.mean():.4f}")
    print(f"  Std:  {rg.std():.4f}")
    print(f"  Non-zero pixels: {(rg > 0).sum():,} of {len(rg):,}")

# Correlation between window configurations (on burned pixels only)
main_rg = results["12-month (main)"]
mask = main_rg > 0  # burned pixels only

print("\n" + "=" * 60)
print("Pearson correlation between window configurations (burned pixels only)")
print("=" * 60)
comparison = []
for label, rg in results.items():
    if label == "12-month (main)":
        continue
    r, _ = pearsonr(rg[mask], main_rg[mask])
    s, _ = spearmanr(rg[mask], main_rg[mask])
    print(f"{label:20s}: Pearson={r:.4f}, Spearman={s:.4f}")
    comparison.append({
        "config": label,
        "vs_main_pearson": r,
        "vs_main_spearman": s,
        "mean": rg[mask].mean(),
        "std":  rg[mask].std(),
    })

df_rg = pd.DataFrame(comparison)
out_path = os.path.join(THESIS_ROOT, "outputs/tables/recovery_gap_sensitivity.csv")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_rg.to_csv(out_path, index=False)
print(f"\n Saved: {out_path}")


# ============================================================
# 2. Composite RIS weighting sensitivity
# ============================================================
print("\n" + "=" * 60)
print("2. Composite RIS weighting sensitivity")
print("=" * 60)

# Load model predictions on test set
preds_path = os.path.join(THESIS_ROOT, "models/main/predictions_test.npz")
if not os.path.exists(preds_path):
    print(f"ERROR: predictions file not found at {preds_path}")
    print("Run Notebook 05 first to generate predictions_test.npz")
    sys.exit(1)

preds = np.load(preds_path)
print(f"Predictions keys: {list(preds.files)}")

# Build a prediction dataframe
n = len(preds[preds.files[0]])
pred_df = pd.DataFrame({
    "severity":     preds["severity"],
    "persistence":  preds["persistence"],
    "recovery_gap": preds["recovery_gap"],
    "recurrence":   preds["recurrence"] if "recurrence" in preds.files else np.zeros(n),
})
print(f"Predictions shape: {pred_df.shape}")

# Five weight profiles
profiles = {
    "equal":                   {"severity": 0.25, "persistence": 0.25, "recovery_gap": 0.25, "recurrence": 0.25},
    "severity_dominant":       {"severity": 0.50, "persistence": 0.20, "recovery_gap": 0.20, "recurrence": 0.10},
    "persistence_dominant":    {"severity": 0.20, "persistence": 0.50, "recovery_gap": 0.20, "recurrence": 0.10},
    "recovery_gap_dominant":   {"severity": 0.20, "persistence": 0.20, "recovery_gap": 0.50, "recurrence": 0.10},
    "recurrence_dominant":     {"severity": 0.20, "persistence": 0.20, "recovery_gap": 0.10, "recurrence": 0.50},
}

# Compute RIS for each profile
ris_scores = {}
for name, w in profiles.items():
    ris = (w["severity"]     * pred_df["severity"]
        +  w["persistence"]  * pred_df["persistence"]
        +  w["recovery_gap"] * pred_df["recovery_gap"]
        +  w["recurrence"]   * pred_df["recurrence"])
    ris_scores[name] = ris.values
    print(f"{name:24s}: mean={ris.mean():.4f}, std={ris.std():.4f}")

# Pairwise Jaccard overlap of top-10% pixels
print("\n" + "=" * 60)
print("Pairwise Jaccard overlap of top-10% pixels")
print("=" * 60)

def top_k_indices(scores, k_frac=0.10):
    """Return indices of top k_frac (e.g., 10%) of pixels."""
    n = len(scores)
    k = int(n * k_frac)
    return set(np.argsort(scores)[-k:])

top_sets = {name: top_k_indices(scores) for name, scores in ris_scores.items()}

names = list(profiles.keys())
jaccard_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        inter = len(top_sets[a] & top_sets[b])
        union = len(top_sets[a] | top_sets[b])
        jaccard_matrix.loc[a, b] = inter / union if union > 0 else 0.0

print(jaccard_matrix.round(3).to_string())

# Pairwise Spearman rank correlation
print("\n" + "=" * 60)
print("Pairwise Spearman rank correlation of full rankings")
print("=" * 60)

spearman_matrix = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        rho, _ = spearmanr(ris_scores[a], ris_scores[b])
        spearman_matrix.loc[a, b] = rho

print(spearman_matrix.round(3).to_string())

# Save
out_jaccard = os.path.join(THESIS_ROOT, "outputs/tables/composite_weighting_jaccard.csv")
out_spearman = os.path.join(THESIS_ROOT, "outputs/tables/composite_weighting_spearman.csv")
jaccard_matrix.to_csv(out_jaccard)
spearman_matrix.to_csv(out_spearman)
print(f"\n Saved: {out_jaccard}")
print(f" Saved: {out_spearman}")

print("\n" + "=" * 60)
print("DONE. Two sensitivity analyses complete.")
print("=" * 60)

1. Recovery-gap window sensitivity


FileNotFoundError: [Errno 2] No such file or directory: '/root/thesisv4/data/processed/X_scaled.npy'